# Smart Inventory — ML Dataset Preparation

This notebook prepares the feature-engineered dataset for machine learning.

### Goal
Create a clean `df_model` that is ready for:
- Feature selection
- Categorical encoding
- Chronological train/validation/test splitting
- XGBoost demand forecasting

### Important rules
- Target: `Units Sold`
- Exclude `Demand Forecast` because of target leakage.
- Initially exclude `Units Ordered` because its correlation with sales is approximately zero.
- Do **not** use random `train_test_split` for this time-series problem.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Load the feature-engineered dataset

This notebook assumes `02_Feature_Engineering_Demand_Forecasting.ipynb` has been run. To make this notebook independently runnable, the feature engineering is reproduced below.

In [2]:
df = pd.read_csv("../DATA/retail_store_inventory.csv")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Original shape:", df.shape)

Original shape: (73100, 15)


## 2. Recreate calendar features

In [3]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Week_of_Year"] = df["Date"].dt.isocalendar().week.astype(int)

df[[
    "Date", "Year", "Month", "Day",
    "Day_of_Week", "Week_of_Year"
]].head()

,Date,Year,Month,Day,Day_of_Week,Week_of_Year
0,2022-01-01,2022,1,1,5,52
1,2022-01-02,2022,1,2,6,52
2,2022-01-03,2022,1,3,0,1
3,2022-01-04,2022,1,4,1,1
4,2022-01-05,2022,1,5,2,1


## 3. Recreate lag and rolling features

In [4]:
group_cols = ["Store ID", "Product ID"]

df["Lag_1"] = df.groupby(group_cols)["Units Sold"].shift(1)
df["Lag_7"] = df.groupby(group_cols)["Units Sold"].shift(7)
df["Lag_14"] = df.groupby(group_cols)["Units Sold"].shift(14)
df["Lag_28"] = df.groupby(group_cols)["Units Sold"].shift(28)

df["Rolling_Mean_7"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df["Rolling_Mean_14"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(14).mean())
)

df["Rolling_Mean_28"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(28).mean())
)

df["Rolling_Std_7"] = (
    df.groupby(group_cols)["Units Sold"]
      .transform(lambda x: x.shift(1).rolling(7).std())
)

df["Price_Difference"] = (
    df["Price"] - df["Competitor Pricing"]
)

print("Feature engineering recreated successfully!")

Feature engineering recreated successfully!


## 4. Remove rows without enough history

The first observations of every Store + Product series do not have enough historical data for the 28-day lag and rolling features. These rows cannot be used for our supervised forecasting model.

In [5]:
time_series_features = [
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_28",
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_28",
    "Rolling_Std_7"
]

df_model = df.dropna(
    subset=time_series_features
).copy()

print("Original rows:", len(df))
print("Rows after feature engineering:", len(df_model))
print("Rows removed:", len(df) - len(df_model))
print("New shape:", df_model.shape)

Original rows: 73100
Rows after feature engineering: 70300
Rows removed: 2800
New shape: (70300, 29)


## 5. Verify that no required feature contains NaN

In [6]:
print("Missing values in time-series features:")
print(df_model[time_series_features].isnull().sum())

print("\nTotal missing values in df_model:")
print(df_model.isnull().sum().sum())

Missing values in time-series features:
Lag_1              0
Lag_7              0
Lag_14             0
Lag_28             0
Rolling_Mean_7     0
Rolling_Mean_14    0
Rolling_Mean_28    0
Rolling_Std_7      0
dtype: int64

Total missing values in df_model:
0


## 6. Define the target

The model will predict `Units Sold`.

In [7]:
target = "Units Sold"

y = df_model[target]

print("Target:", target)
print("Target shape:", y.shape)
print("\nTarget statistics:")
print(y.describe())

Target: Units Sold
Target shape: (70300,)

Target statistics:
count    70300.000000
mean       136.500626
std        109.018835
min          0.000000
25%         48.000000
50%        107.000000
75%        203.000000
max        499.000000
Name: Units Sold, dtype: float64


## 7. Select initial model features

### Excluded
- `Demand Forecast`: target leakage
- `Units Ordered`: initially excluded because its correlation with `Units Sold` is approximately zero
- `Date`: the raw date is not directly passed to the model; calendar features are used instead

### Retained
Store/product identity, business variables, calendar features, and historical demand features.

In [13]:
feature_columns = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Inventory Level",
    "Price",
    "Discount",
    "Weather Condition",
    "Holiday/Promotion",
    "Competitor Pricing",
    "Seasonality",

    # Calendar features
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
    "Week_of_Year",

    # Lag features
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_28",

    # Rolling demand features
    "Rolling_Mean_7",
    "Rolling_Mean_14",
    "Rolling_Mean_28",
    "Rolling_Std_7",

    # Price feature
    "Price_Difference"
]

X = df_model[feature_columns].copy()

print("Number of features:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

Number of features: 25
X shape: (70300, 25)
y shape: (70300,)


,Store ID,Product ID,Category,Region,Inventory Level,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality,Year,Month,Day,Day_of_Week,Week_of_Year,Lag_1,Lag_7,Lag_14,Lag_28,Rolling_Mean_7,Rolling_Mean_14,Rolling_Mean_28,Rolling_Std_7,Price_Difference
28,S001,P0001,Furniture,South,154,88.06,15,Snowy,1,92.84,Autumn,2022,1,29,5,4,97.0,191.0,176.0,127.0,111.142857,137.357143,116.071429,71.601742,-4.78
29,S001,P0001,Furniture,West,498,68.76,10,Snowy,0,73.29,Spring,2022,1,30,6,4,47.0,79.0,97.0,81.0,90.571429,128.142857,113.214286,65.237661,-4.53
30,S001,P0001,Clothing,West,410,70.78,0,Sunny,1,68.88,Summer,2022,1,31,0,5,130.0,9.0,69.0,5.0,97.857143,130.500000,114.964286,66.564326,1.90
31,S001,P0001,Groceries,East,419,34.49,0,Sunny,0,33.85,Summer,2022,2,1,1,5,200.0,189.0,123.0,58.0,125.142857,139.857143,121.928571,63.127988,0.64
32,S001,P0001,Electronics,South,415,52.49,5,Cloudy,0,48.25,Autumn,2022,2,2,2,5,279.0,49.0,211.0,147.0,138.000000,151.000000,129.821429,84.011904,4.24


## 8. Check feature data types

We will identify categorical and numerical features before encoding.

In [14]:
categorical_features = X.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['Store ID', 'Product ID', 'Category', 'Region', 'Weather Condition', 'Seasonality']

Numerical features:
['Inventory Level', 'Price', 'Discount', 'Holiday/Promotion', 'Competitor Pricing', 'Year', 'Month', 'Day', 'Day_of_Week', 'Week_of_Year', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_28', 'Rolling_Std_7', 'Price_Difference']


## 9. Check cardinality of categorical variables

This helps us decide how to encode each categorical feature.

In [15]:
for col in categorical_features:
    print(f"{col}: {X[col].nunique()} unique values")
    print(X[col].unique())
    print()

Store ID: 5 unique values
<StringArray>
['S001', 'S002', 'S003', 'S004', 'S005']
Length: 5, dtype: str

Product ID: 20 unique values
<StringArray>
['P0001', 'P0002', 'P0003', 'P0004', 'P0005', 'P0006', 'P0007', 'P0008',
 'P0009', 'P0010', 'P0011', 'P0012', 'P0013', 'P0014', 'P0015', 'P0016',
 'P0017', 'P0018', 'P0019', 'P0020']
Length: 20, dtype: str

Category: 5 unique values
<StringArray>
['Furniture', 'Clothing', 'Groceries', 'Electronics', 'Toys']
Length: 5, dtype: str

Region: 4 unique values
<StringArray>
['South', 'West', 'East', 'North']
Length: 4, dtype: str

Weather Condition: 4 unique values
<StringArray>
['Snowy', 'Sunny', 'Cloudy', 'Rainy']
Length: 4, dtype: str

Seasonality: 4 unique values
<StringArray>
['Autumn', 'Spring', 'Summer', 'Winter']
Length: 4, dtype: str



## 10. Verify target leakage is not present

In [16]:
print("Demand Forecast in X:", "Demand Forecast" in X.columns)
print("Units Sold in X:", "Units Sold" in X.columns)

assert "Demand Forecast" not in X.columns
assert "Units Sold" not in X.columns

print("Leakage check passed!")

Demand Forecast in X: False
Units Sold in X: False
Leakage check passed!


## 11. Sort model data chronologically

The final split must preserve time order. We will not randomly shuffle observations.

In [17]:
model_data = df_model.sort_values(
    "Date"
).reset_index(drop=True)

print("Start date:", model_data["Date"].min())
print("End date:", model_data["Date"].max())
print("Rows:", len(model_data))

model_data[[
    "Date", "Store ID", "Product ID", "Units Sold"
]].head()

Start date: 2022-01-29 00:00:00
End date: 2024-01-01 00:00:00
Rows: 70300


,Date,Store ID,Product ID,Units Sold
0,2022-01-29,S001,P0001,47
1,2022-01-29,S001,P0013,157
2,2022-01-29,S001,P0016,34
3,2022-01-29,S004,P0015,218
4,2022-01-29,S002,P0019,72


## 12. Next step

The dataset is now ready for the **encoding + chronological split** stage.

Recommended split:

```text
2022 ───────────────── 2023 ───────────── 2024
       Training              Validation/Test
```

Before training XGBoost, we will:
1. Create a chronological train/validation/test split.
2. Encode categorical variables without leaking information from the future.
3. Keep the target separate.
4. Train a simple baseline.
5. Train XGBoost.
6. Evaluate MAE, RMSE and R².

**Do not use random `train_test_split`.**